In [0]:
from pyspark.sql import functions as F

DIM_DATE_TABLE = "nyc_taxi.gold.dim_date"
DAILY_SUMMARY_TABLE = "nyc_taxi.gold.daily_summary"
HOURLY_DEMAND_TABLE = "nyc_taxi.gold.hourly_demand"
ZONE_PERFORMANCE_TABLE = "nyc_taxi.gold.zone_performance"
SILVER_TABLE = "nyc_taxi.silver.trips_clean"

In [0]:
%sql
SELECT 'dim_date' as table_name, COUNT(*) as rows_count FROM nyc_taxi.gold.dim_date
UNION ALL
SELECT 'daily_summary' as table_name, COUNT(*) as rows_count FROM nyc_taxi.gold.daily_summary
UNION ALL
SELECT 'hourly_demand' as table_name, COUNT(*) as rows_count FROM nyc_taxi.gold.hourly_demand
UNION ALL
SELECT 'zone_performance' as table_name, COUNT(*) as rows_count FROM nyc_taxi.gold.zone_performance

table_name,rows_count
dim_date,91
daily_summary,91
hourly_demand,168
zone_performance,261


In [0]:
%sql
SELECT
    table_name,
    column_name,
    data_type,
    ordinal_position
FROM nyc_taxi.information_schema.columns
WHERE table_schema = 'gold'
  AND table_name IN ('dim_date', 'daily_summary', 'hourly_demand', 'zone_performance')
ORDER BY table_name, ordinal_position;

table_name,column_name,data_type,ordinal_position
daily_summary,pickup_date,DATE,0
daily_summary,date_key,INT,1
daily_summary,total_trips,LONG,2
daily_summary,total_revenue,DOUBLE,3
daily_summary,avg_fare,DOUBLE,4
daily_summary,avg_tip,DOUBLE,5
daily_summary,avg_trip_distance,DOUBLE,6
daily_summary,avg_trip_duration_minutes,DOUBLE,7
daily_summary,unique_pickup_zones,LONG,8
dim_date,date_key,INT,0


In [0]:
%sql
SELECT
    (SELECT SUM(total_trips) FROM nyc_taxi.gold.daily_summary) AS gold_total_trips,
    (SELECT COUNT(*) FROM nyc_taxi.silver.trips_clean) AS silver_total_trips;

gold_total_trips,silver_total_trips
11634221,11634221


In [0]:
%sql
SELECT
    (SELECT COUNT(*) FROM nyc_taxi.gold.daily_summary WHERE total_revenue < 0) AS negative_revenue_days,
    (SELECT COUNT(*) FROM nyc_taxi.gold.zone_performance WHERE zone_name IS NULL) AS null_zone_names,
    (SELECT COUNT(*) FROM nyc_taxi.gold.zone_performance WHERE total_trips <= 0) AS zero_trips_zone,
    (SELECT COUNT(*) FROM nyc_taxi.gold.daily_summary AS ds
    LEFT JOIN nyc_taxi.gold.dim_date dd
    ON ds.date_key = dd.date_key
    WHERE dd.date_key IS NULL) AS orphan_dates;

negative_revenue_days,null_zone_names,zero_trips_zone,orphan_dates
0,0,0,0
